# Sentiment Analysis Using Traditional Machine Learning

## Task 1 – Download and Load the Dataset

In [ ]:

import pandas as pd
from pathlib import Path 

dataset_path = Path("../dataset")

# Load the datasets
amazon = pd.read_csv(
    dataset_path / "amazon_cells_labelled.txt",
    sep="\t",
    header=None,
    names=["text", "label"]
)

imdb = pd.read_csv(
    dataset_path / "imdb_labelled.txt",
    sep="\t",
    header=None,
    names=["text", "label"]
)

yelp = pd.read_csv(
    dataset_path / "yelp_labelled.txt",
    sep="\t",
    header=None,
    names=["text", "label"]
)

# Combine the datasets
df = pd.concat([amazon, imdb, yelp], ignore_index=True)

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (2748, 2)


,text,label
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


In [16]:

print("Amazon reviews:", len(amazon))
print("IMDb reviews:", len(imdb))
print("Yelp reviews:", len(yelp))

print("\nTotal reviews:", len(df))


Amazon reviews: 1000
IMDb reviews: 748
Yelp reviews: 1000

Total reviews: 2748



### Dataset Overview

The dataset contains 2,748 labelled reviews collected from three sources:

- Amazon: 1,000 reviews
- IMDb: 748 reviews
- Yelp: 1,000 reviews

Each review contains a sentiment label:
- 0: Negative
- 1: Positive


## Task 2 – Text Preprocessing

In [17]:

import re

def clean_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove punctuation and unnecessary characters
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra whitespace
    text = " ".join(text.split())

    return text

# Create a new column for cleaned text
df["clean_text"] = df["text"].apply(clean_text)

# Display original and cleaned text
display(df[["text", "clean_text"]].head(10))


,text,clean_text
0,So there is no way for me to plug it in here i...,so there is no way for me to plug it in here i...
1,"Good case, Excellent value.",good case excellent value
2,Great for the jawbone.,great for the jawbone
3,Tied to charger for conversations lasting more...,tied to charger for conversations lasting more...
4,The mic is great.,the mic is great
5,I have to jiggle the plug to get it to line up...,i have to jiggle the plug to get it to line up...
6,If you have several dozen or several hundred c...,if you have several dozen or several hundred c...
7,If you are Razr owner...you must have this!,if you are razr owner you must have this
8,"Needless to say, I wasted my money.",needless to say i wasted my money
9,What a waste of money and time!.,what a waste of money and time


In [18]:

print("Dataset shape:", df.shape)

print("\nOriginal text:")
print(df["text"].iloc[1])

print("\nCleaned text:")
print(df["clean_text"].iloc[1])


Dataset shape: (2748, 3)

Original text:
Good case, Excellent value.

Cleaned text:
good case excellent value


## Task 3 – Explore the Text Data

In [19]:

# Positive reviews (label = 1)
print("Positive Reviews:")
display(df[df["label"] == 1][["text", "label"]].head(5))

# Negative reviews (label = 0) 
print("Negative Reviews:")
display(df[df["label"] == 0][["text", "label"]].head(5))

Positive Reviews:


,text,label
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
4,The mic is great.,1
7,If you are Razr owner...you must have this!,1
10,And the sound quality is great.,1


Negative Reviews:


,text,label
0,So there is no way for me to plug it in here i...,0
3,Tied to charger for conversations lasting more...,0
5,I have to jiggle the plug to get it to line up...,0
6,If you have several dozen or several hundred c...,0
8,"Needless to say, I wasted my money.",0


In [20]:

from collections import Counter

# Separate positive and negative cleaned reviews
positive_text = " ".join(df[df["label"] == 1]["clean_text"])
negative_text = " ".join(df[df["label"] == 0]["clean_text"])

# Count word frequencies
positive_words = Counter(positive_text.split())
negative_words = Counter(negative_text.split())

# 10 most frequent words
print("Most Frequent Words in Positive Reviews:")
print(positive_words.most_common(10))

print("\nMost Frequent Words in Negative Reviews:")
print(negative_words.most_common(10))

Most Frequent Words in Positive Reviews:
[('the', 960), ('and', 665), ('i', 475), ('a', 449), ('is', 397), ('it', 363), ('this', 318), ('to', 288), ('of', 287), ('was', 248)]

Most Frequent Words in Negative Reviews:
[('the', 993), ('i', 562), ('and', 473), ('a', 440), ('it', 426), ('to', 382), ('is', 357), ('of', 337), ('this', 325), ('was', 323)]


## Task 4 – Split the Dataset

In [21]:

from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["label"]

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 2198
Testing samples: 550


## Task 5 – Create Bag of Words (BoW) Representation

In [22]:

from sklearn.feature_extraction.text import CountVectorizer

# Create Bag of Words vectorizer
vectorizer = CountVectorizer()

# Convert training and testing text into numerical features
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

# Display the shapes
print("Training BoW shape:", X_train_bow.shape)
print("Testing BoW shape:", X_test_bow.shape)

Training BoW shape: (2198, 4517)
Testing BoW shape: (550, 4517)


## Task 6 – Train a Machine Learning Model

In [23]:

from sklearn.linear_model import LogisticRegression

# Create the Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)

# Train the model using Bag of Words features
model.fit(X_train_bow, y_train)

print("Model training completed successfully!")       

Model training completed successfully!


## Task 7 – Evaluate the Model's Performance

In [24]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Predict sentiments using the test data
y_pred = model.predict(X_test_bow)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Display the results
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

# Display detailed Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred)) 

Accuracy: 0.8218
Precision: 0.8403
Recall: 0.7978
F1-score: 0.8185

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.85      0.82       273
           1       0.84      0.80      0.82       277

    accuracy                           0.82       550
   macro avg       0.82      0.82      0.82       550
weighted avg       0.82      0.82      0.82       550



## Task 8 – Test the Model with a New Sentence

In [25]:

# New sentence
new_sentence = ["This product is amazing and works perfectly!"]

# Preprocess the new sentence
cleaned_sentence = [clean_text(new_sentence[0])]

# Convert the sentence into BOW features
new_sentence_bow = vectorizer.transform(cleaned_sentence)

# Predict the sentiment
prediction = model.predict(new_sentence_bow)

# Display the result
print("Sentence:", new_sentence[0])

if prediction[0] == 1:
    print("Predicted Sentiment: Positive")
else:
    print("Predicted Sentiment: Negative") 

Sentence: This product is amazing and works perfectly!
Predicted Sentiment: Positive
